# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
from scipy.stats import levene
import statsmodels.stats.multitest as smm

from utilis import *
import sys
sys.path.append('../')
from settings import *

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
SAVE_DIR = 'figures/'
df = pd.read_excel(DATA_PATH, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head(1)

In [ ]:
df.rename(columns={'#':'ID',
                   'תאריך בדיקות דם': 'blood_test_date',
                   'שעת לקיחת דם':'blood_taken_time',
                   'מספר מנות':'num_of_injections',
                   'גיל הפסקת מחזור':'menopause',
                   'אסטרוגן':'estorgen',
                   'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                   'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                   'p-tau_ new_#1':'p-tau_new_#1',
                   'ביקורת אחרי שנה':'checkup after a year',
                   'חיסון':'vaccine',
                   'תרופות':'medications',
                   'הערות':'notes',
                   'Number_of_risk_faktors':'Number_of_risk_factors',
                   'P-tau217':'p-tau217',
                   'צרבונין':'Cerebonin',
                   'Spatial ': 'Spatial'
                   }, inplace=True)

In [ ]:
df = df[df['MOCA_score'] != '?']

In [ ]:
df = df.apply(pd.to_numeric, errors='ignore')

In [ ]:
df['MCI'] = df[MOCA_COL] < MCI_THR
((df['MCI'] == True) & (df['p-tau217'] < HIGH_TAU_217_THR)).sum(), ((df['MCI'] == True) & (df['p-tau217'] > HIGH_TAU_217_THR)).sum()


# Outliers

In [ ]:
df.iloc[df[TAU_217].argmax()]
outliers = []  # Participant exclusions are maintained privately.

print(df.shape) 
df = df.drop(df[df['ID'].isin(outliers)].index)
print(df.shape) 

In [ ]:
(df['p-tau217'] > HIGH_TAU_217_THR).value_counts()

# Add columns

In [ ]:
df = df[df[MOCA_COL] != '?']
df['MCI'] = df[MOCA_COL] < MCI_THR
df[TAU_CAT] = pd.cut(df[TAU_217], bins=[0, INT_LOW_TAU_217_THR, INT_HIGH_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=['Low', 'Intermediate_low', 'Intermediate_high', 'High'])
df_full = df[df[MOCA_COL].notna()].copy()

df = df[df[TAU_217].notna()] # Filter only to those with tau_217 values
df = df[df[MOCA_COL] > MIN_MOCA_SCORE] # removing samples with moca below the minimum threshold of 19
df_high_vs_low = df[(df[TAU_217] > HIGH_TAU_217_THR) | (df[TAU_217] < INT_LOW_TAU_217_THR)]
df_high_vs_low[TAU_CAT] = df_high_vs_low[TAU_CAT].cat.remove_categories(['Intermediate_low', 'Intermediate_high'])
    
df_high_vs_low_mci = df_high_vs_low[df_high_vs_low['MCI']]
df_high_vs_low_mci_high = df_high_vs_low_mci[df_high_vs_low_mci[TAU_CAT] == 'High']
df_high_vs_low_mci_low = df_high_vs_low_mci[df_high_vs_low_mci[TAU_CAT] == 'Low']

df_high_vs_low_mci_high_moca = df_high_vs_low_mci_high[df_high_vs_low_mci_high[MOCA_COL] > MCI_THR]
df_high_vs_low_mci_low_moca = df_high_vs_low_mci_low[df_high_vs_low_mci_low[MOCA_COL] > MCI_THR]

In [ ]:
df_high_vs_low[TAU_CAT].value_counts()

## Cognitive Tests

In [ ]:
COGNITIVE_SAVE_DIR = SAVE_DIR + 'cognitive/'

Filter cognitive damaged subjects

In [ ]:
# df_high_vs_low = df_high_vs_low[df_high_vs_low['MCI'] == True]
df_high_vs_low[TAU_CAT].value_counts()

### Executive

In [ ]:
EXECTUTIVE = 'Executive'

df_high_vs_low_exe = df_high_vs_low[df_high_vs_low[EXECTUTIVE].notna()]
graph_colors = [healthy_color, risk_color]
kdeplot_high_vs_low(df_high_vs_low_exe, EXECTUTIVE, TAU_CAT, graph_colors, 'Executive vs Tau 217 category',COGNITIVE_SAVE_DIR + 'Executive_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_exe, EXECTUTIVE, TAU_CAT, 4, graph_colors, 'Executive vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Executive_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_exe[df_high_vs_low_exe[TAU_CAT] == 'Low'][EXECTUTIVE]
group2 = df_high_vs_low_exe[df_high_vs_low_exe[TAU_CAT] == 'High'][EXECTUTIVE]

p_executive = stat_test(data_type, group1, group2)

### Spatial

In [ ]:
SPATIAL = 'Spatial'
df_high_vs_low_spatial = df_high_vs_low[df_high_vs_low[SPATIAL].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_spatial, SPATIAL, TAU_CAT, graph_colors, 'Spatial vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Spatial_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_spatial, SPATIAL, TAU_CAT, 4, graph_colors, 'Spatial vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Spatial_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_spatial[df_high_vs_low_spatial[TAU_CAT] == 'Low'][SPATIAL]
group2 = df_high_vs_low_spatial[df_high_vs_low_spatial[TAU_CAT] == 'High'][SPATIAL]

p_spatial = stat_test(data_type, group1, group2)

### Attention_and_concentration

In [ ]:
A_C = 'Attention_and_concentration'
df_high_vs_low_ac = df_high_vs_low[df_high_vs_low[A_C].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_ac, A_C, TAU_CAT, graph_colors, 'Attention/Concentration vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Attention_Concentration_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_ac, A_C, TAU_CAT, 5, graph_colors, 'Attention/Concentration vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Attention_Concentration_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Ordinal'
group1 = df_high_vs_low_ac[df_high_vs_low_ac[TAU_CAT] == 'Low'][A_C]
group2 = df_high_vs_low_ac[df_high_vs_low_ac[TAU_CAT] == 'High'][A_C]

p_attention = stat_test(data_type, group1, group2)

### Memory

In [ ]:
MEMORY = 'Memory'
df_high_vs_low_memory = df_high_vs_low[df_high_vs_low[MEMORY].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_memory, MEMORY, TAU_CAT, graph_colors, 'Memory vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Memory_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_memory, MEMORY, TAU_CAT, 5, graph_colors, 'Memory vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Memory_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_memory[df_high_vs_low_memory[TAU_CAT] == 'Low'][MEMORY]
group2 = df_high_vs_low_memory[df_high_vs_low_memory[TAU_CAT] == 'High'][MEMORY]

p_memory = stat_test(data_type, group1, group2)

### Language

In [ ]:
LANGUAGE = 'Language'
df_high_vs_low_language = df_high_vs_low[df_high_vs_low[LANGUAGE].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_language, LANGUAGE, TAU_CAT, graph_colors, 'Language vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Language_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_language, LANGUAGE, TAU_CAT, 5, graph_colors, 'Language vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Language_vs_Tau_217_category_hist.pdf')

In [ ]:
# statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_language[df_high_vs_low_language[TAU_CAT] == 'Low'][LANGUAGE]
group2 = df_high_vs_low_language[df_high_vs_low_language[TAU_CAT] == 'High'][LANGUAGE]

p_language = stat_test(data_type, group1, group2)

### Orientation

In [ ]:
ORIENTATION = 'Orientation'

df_high_vs_low_orientation = df_high_vs_low[df_high_vs_low[ORIENTATION].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_orientation, ORIENTATION, TAU_CAT, graph_colors, 'Orientation vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Orientation_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_orientation, ORIENTATION, TAU_CAT, 6, graph_colors, 'Orientation vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Orientation_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'

# group1 = df_high_vs_low_orientation
# group2 = [TAU_CAT, ORIENTATION]

group1 = df_high_vs_low_orientation[df_high_vs_low_orientation[TAU_CAT] == 'Low'][ORIENTATION]
group2 = df_high_vs_low_orientation[df_high_vs_low_orientation[TAU_CAT] == 'High'][ORIENTATION]


p_orientation = stat_test(data_type, group1, group2) # Too few samples for the test

### Frontal memory score (#1)

In [ ]:
FMC = 'Frontal memory score (#1)'

df_high_vs_low_fmc = df_high_vs_low[df_high_vs_low[FMC].notna()]
df_high_vs_low_fmc[FMC] = df_high_vs_low_fmc[FMC].astype(float)
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_fmc, FMC, TAU_CAT, graph_colors, 'Frontal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Frontal_Memory_Score_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_fmc, FMC, TAU_CAT, 4, graph_colors, 'Frontal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Frontal_Memory_Score_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_fmc[df_high_vs_low_fmc[TAU_CAT] == 'Low'][FMC]
group2 = df_high_vs_low_fmc[df_high_vs_low_fmc[TAU_CAT] == 'High'][FMC]

p_frontal_memory = stat_test(data_type, group1, group2)

### Hipocampal memory score (#2)

In [ ]:
HMC = 'Hipocampal memory score (#2)'

df_high_vs_low_hmc = df_high_vs_low[df_high_vs_low[HMC].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_hmc, HMC, TAU_CAT, graph_colors, 'Hipocampal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Hipocampal_Memory_Score_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_hmc, HMC, TAU_CAT, 5, graph_colors, 'Hipocampal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Hipocampal_Memory_Score_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_hmc[df_high_vs_low_hmc[TAU_CAT] == 'Low'][HMC]
group2 = df_high_vs_low_hmc[df_high_vs_low_hmc[TAU_CAT] == 'High'][HMC]

p_hipocampal_memory = stat_test(data_type, group1, group2)

### MOCA score 

In [ ]:
df_high_vs_low.columns

In [ ]:
MOCA_COL = 'MOCA_score'

df_high_vs_low_moca = df_high_vs_low[df_high_vs_low[MOCA_COL].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_moca, MOCA_COL, TAU_CAT, graph_colors, 'MOCA score Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'MOCA score_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_moca, MOCA_COL, TAU_CAT, 5, graph_colors, 'MOCA score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'MOCA score_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low[df_high_vs_low[TAU_CAT] == 'Low'][MOCA_COL].astype(int)
group2 = df_high_vs_low[df_high_vs_low[TAU_CAT] == 'High'][MOCA_COL].astype(int)

p_moca = stat_test(data_type, group1, group2)

# Statistical Correction
I'm using BH-FDR

In [ ]:
ALPHA = 0.05
METHOD = 'fdr_bh'

In [ ]:
df_high_vs_low[TAU_CAT].value_counts()

In [ ]:
# Statistical Correction
p_values = [p_attention, p_executive, p_spatial, p_memory, p_language, p_orientation, p_frontal_memory, p_hipocampal_memory, p_moca]
cognitive_tests = ['Attention and concentration', 'Executive', 'Spatial', 'Memory', 'Language', 'Orientation', 'Frontal memory score', 'Hipocampal memory score', 'MOCA score']
p_values = [p for p in p_values if p is not None]
reject, p_values_corrected, _, _ = smm.multipletests(p_values, alpha=ALPHA, method=METHOD)

for i, p in enumerate(p_values):
    if p is not None:
        print(f'{cognitive_tests[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

print("\nSignificant features:")
for i, p in enumerate(p_values):
    if p < ALPHA:
        print(f'{cognitive_tests[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

In [ ]:
HIGH_TAU_217_THR

# Results
### 0.53
* After statistical correction there are no signifcant cognitive tests.
* Memory, and frontal memory score where significant with 0.046, 0.029 and after correction 0.186, 0.186 respectivly
* MOCA score: p-value = 0.017 -> adjusted p-value = 0.130

### 0.449
* No features where signifcant after correction
* MOCA score: p-value = 0.032 -> adjusted p-value = 0.281

# Results visualization

### IST

In [ ]:
df_high_vs_low_ist = df_high_vs_low.dropna(subset=[IST_COL])
df_high_vs_low_ist[IST_COL] = df_high_vs_low_ist[IST_COL].astype(float)
grpah_colors = [healthy_color, risk_color]

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_ist[df_high_vs_low_ist[TAU_CAT] == 'Low'][IST_COL]
group2 = df_high_vs_low_ist[df_high_vs_low_ist[TAU_CAT] == 'High'][IST_COL]

p_ist = stat_test(data_type, group1, group2)

### TMT (Trail B)

In [ ]:
df_high_vs_low_tmt = df_high_vs_low.dropna(subset=[TMT_B_COL])
df_high_vs_low_tmt[TMT_B_COL] = df_high_vs_low_tmt[TMT_B_COL].astype(float)
graph_colors = [healthy_color, risk_color]

In [ ]:
# Statistical test
data_type = 'Continuous'
group1 = df_high_vs_low_tmt[df_high_vs_low_tmt[TAU_CAT] == 'Low'][TMT_B_COL]
group2 = df_high_vs_low_tmt[df_high_vs_low_tmt[TAU_CAT] == 'High'][TMT_B_COL]

p_trail_b = stat_test(data_type, group1, group2)

### MBI-C

In [ ]:
df_high_vs_low_mbi = df_high_vs_low.dropna(subset=[MBI_C_COL])
df_high_vs_low_mbi[MBI_C_COL] = df_high_vs_low_mbi[MBI_C_COL].astype(float)
grpah_colors = [healthy_color, risk_color]

In [ ]:
kdeplot_high_vs_low(df_high_vs_low_mbi, MBI_C_COL, TAU_CAT, graph_colors, 'MBI-C vs Tau 217 category','', alpha=0.5, save=False)

In [ ]:
# threshold MBI-C
MBI_C_THR = 6
df_high_vs_low_mbi['MBI-C_THR'] = (df_high_vs_low_mbi[MBI_C_COL] > MBI_C_THR)
# df_high_vs_low_mbi_c_thr['MBI-C_THR']

df_high_vs_low_mbi_c_thr = df_high_vs_low_mbi.dropna(subset=['MBI-C_THR'])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_mbi_c_thr, 'MBI-C_THR', TAU_CAT, 2 , graph_colors, 'MBI-C_THR vs Tau 217 category', '', save=False)

In [ ]:
# Statistical test

data_type = 'Ordinal'

group1 = df_high_vs_low_mbi[df_high_vs_low_mbi[TAU_CAT] == 'Low'][MBI_C_COL]
group2 = df_high_vs_low_mbi[df_high_vs_low_mbi[TAU_CAT] == 'High'][MBI_C_COL]

p_mbi_c = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_mbi
group2 = [TAU_CAT, 'MBI-C_THR']

p_mbi_c_thr = stat_test(data_type, group1, group2) # Too few samples for the test

### log p-value heatmap

In [ ]:
p_values += [p_trail_b, p_ist, p_mbi_c]
cognitive_tests += ['Trail B', 'IST', 'MBI-C']

In [ ]:
cognitive_tests_names = ['Attention_and_concentration', 'Executive', 'Spatial',
       'Memory', 'Language', 'Orientation', 'Frontal memory score (#1)',
       'Hipocampal memory score (#2)', MOCA_COL,
       'Trail_B_(second)', 'IST', 'Mild_behavioral_Impairment_Checklist']
change_direction = []
for c_test in cognitive_tests_names:
    diff = df_high_vs_low[df_high_vs_low[TAU_CAT] == 'High'][c_test].mean() - df_high_vs_low[df_high_vs_low[TAU_CAT] == 'Low'][c_test].mean()
    print(f'{c_test}: {diff:.2f}')
    change_direction.append(diff)

change_direction = np.array(change_direction)
change_direction = np.sign(change_direction)

In [ ]:
# plot heatmap of cognitive tests. The color is based on log_p_values
p_values = np.array(p_values)
log_p_values = np.log(p_values)

p_values_corrected = np.array(p_values_corrected)

signed_log_p_values = change_direction * log_p_values

fig, ax = plt.subplots(figsize=(10, 2))
sns.heatmap((signed_log_p_values).reshape(1, -1), annot=False, fmt='.2f', cmap='coolwarm', cbar=True, ax=ax)
ax.set_xticklabels(cognitive_tests, rotation=45, ha='right')
ax.set_yticklabels(['log p-value * sign of change'], rotation=0, ha='right')
plt.title('Cognitive tests vs Tau 217 category')
plt.savefig(COGNITIVE_SAVE_DIR + 'cognitivie healthy-Cognitive_tests_vs_Tau_217_category_heatmap.pdf')
plt.show()

In [ ]:
log_p_values

# Liat's data

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/clinical_data_analysis/Tau_217/ADPD presentation p-tau217 26.3.25.xlsx'
xls = pd.ExcelFile(DATA_PATH)

# Get sheet names
sheet_names = xls.sheet_names
sheet_names

In [ ]:
l_df = pd.read_excel('/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/clinical_data_analysis/Tau_217/ADPD presentation p-tau217 26.3.25.xlsx', sheet_name='AD MCI versus non-AD MCI')
l_df = l_df[1:]
l_df.head(2)

In [ ]:
cognitive_tests_names = l_df.Factor
change_direction = []
for c_test in cognitive_tests_names:
    diff = (l_df['AD MCI'][l_df['Factor'] == c_test] - l_df['NON-AD MCI'][l_df['Factor'] == c_test]).values[0]
    print(f'{c_test}: {diff:.2f}')
    change_direction.append(diff)

change_direction = np.array(change_direction)
change_direction = np.sign(change_direction)

# plot heatmap of cognitive tests. The color is based on log_p_values
p_values = np.array(l_df['p-value'])
log_p_values = np.log(p_values)

signed_log_p_values = change_direction * log_p_values

fig, ax = plt.subplots(figsize=(10, 2))
sns.heatmap((signed_log_p_values).reshape(1, -1), annot=False, fmt='.2f', cmap='coolwarm', cbar=True, ax=ax)
ax.set_xticklabels(cognitive_tests_names, rotation=45, ha='right')
ax.set_yticklabels(['log p-value * sign of change'], rotation=0, ha='right')
plt.title('AD MCI vs NON-AD MCI cognitive tests')
plt.savefig(COGNITIVE_SAVE_DIR + 'AD MCI versus non-AD MCI cognitive tests heatmap.pdf', bbox_inches='tight')
plt.show()